## Sentimiento global en reviews (Ollama + Llama 3)

Clasificación del **sentimiento general** de cada review en `validation_dataset.csv` (positivo / negativo / neutro) y comparación con la etiqueta real `sentimiento`.

**Requisitos:** [Ollama](https://ollama.com) en ejecución local, modelo `llama3` (`ollama pull llama3`), y paquetes: `pip install ollama pandas scikit-learn tqdm`.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

try:
    import ollama
except ImportError:
    raise ImportError("Instala: pip install ollama pandas")

# Notebook en Tesis/notebooks/
DATA_PATH = Path(r"../sources/data/validation_dataset.csv").resolve()
OUT_PATH = Path(r"../sources/data/validation_sentimiento_llama3.csv").resolve()

MODELO = "llama3"
ETIQUETAS_VALIDAS = ("positivo", "negativo", "neutro")

print("CSV:", DATA_PATH.exists(), DATA_PATH)
print("Salida:", OUT_PATH)
print("Modelo:", MODELO)

CSV: True C:\Estudio\Maestria\Tesis\sources\data\validation_dataset.csv
Salida: C:\Estudio\Maestria\Tesis\sources\data\validation_sentimiento_llama3.csv
Modelo: llama3


### Prompt y llamada al modelo

Salida **JSON** con un único campo `sentimiento`: la valoración **global** de toda la review. No se envía la etiqueta real al modelo (evita fuga de información).

In [2]:
SYSTEM_PROMPT = """Eres un analista de opiniones de hoteles en español.

Tu tarea: leer UNA review completa y clasificar el SENTIMIENTO GENERAL de la experiencia descrita (toda la review, no un solo aspecto aislado).

Valores permitidos para "sentimiento" (usa exactamente uno):
- "positivo": la experiencia global es claramente favorable o recomendaría la estancia.
- "negativo": la experiencia global es claramente desfavorable, quejas fuertes o no recomendaría.
- "neutro": mezcla equilibrada de lo bueno y lo malo, tono factual sin inclinación clara, o pros y contras que se compensan.

Responde SOLO con un objeto JSON válido, sin markdown ni texto fuera del JSON.
Formato obligatorio:
{"sentimiento": "positivo"}
"""


def _normalizar_sentimiento(val) -> str | None:
    if val is None or (isinstance(val, str) and not val.strip()):
        return None
    s = str(val).strip().lower()
    if s in ("positivo", "positive", "pos"):
        return "positivo"
    if s in ("negativo", "negative", "neg"):
        return "negativo"
    if s in ("neutro", "neutral"):
        return "neutro"
    return None


def parsear_sentimiento_json(texto: str) -> str | None:
    texto = texto.strip()
    m = re.search(r"\{[\s\S]*\}", texto)
    if not m:
        return None
    try:
        data = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    return _normalizar_sentimiento(data.get("sentimiento"))


def extraer_uso_tokens(resp: dict) -> dict:
    p = resp.get("prompt_eval_count")
    e = resp.get("eval_count")
    total = None
    if isinstance(p, int) and isinstance(e, int):
        total = p + e
    return {
        "prompt_eval_count": p,
        "eval_count": e,
        "total_tokens": total,
    }


def predecir_sentimiento(texto_review: str, *, return_usage: bool = False):
    """Clasifica el sentimiento global. No incluye la etiqueta real."""
    user = f"Review:\n{texto_review}"
    r = ollama.chat(
        model=MODELO,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        format="json",
        options={"temperature": 0.2},
    )
    content = r["message"]["content"]
    pred = parsear_sentimiento_json(content)
    if return_usage:
        return pred, extraer_uso_tokens(r)
    return pred

In [3]:
# Iniciar Ollama si no está corriendo (Windows / Unix)
import subprocess
import sys


def iniciar_ollama():
    import socket
    import time

    def is_ollama_running(host="localhost", port=11434):
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            return False

    if not is_ollama_running():
        print("Ollama no está corriendo. Intentando iniciar Ollama...")
        try:
            if sys.platform.startswith("win"):
                subprocess.Popen(
                    "ollama serve",
                    shell=True,
                    creationflags=subprocess.DETACHED_PROCESS,
                )
            else:
                subprocess.Popen(
                    ["ollama", "serve"],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL,
                )
            for _ in range(10):
                if is_ollama_running():
                    print("Ollama iniciado exitosamente.")
                    break
                time.sleep(1)
            else:
                print("No se pudo iniciar Ollama automáticamente. Inícialo manualmente.")
        except Exception as ex:
            print(f"Error al intentar iniciar Ollama: {ex}")


iniciar_ollama()

Ollama no está corriendo. Intentando iniciar Ollama...
Ollama iniciado exitosamente.


### Cargar datos y prueba con una fila

In [4]:
df = pd.read_csv(DATA_PATH)
assert "review" in df.columns and "sentimiento" in df.columns, "Se esperan columnas 'review' y 'sentimiento'"
df["sentimiento"] = df["sentimiento"].astype(str).str.strip().str.lower()
print(df.shape)
print(df["sentimiento"].value_counts())
df.head(3)

(1462, 3)
sentimiento
neutro      674
positivo    400
negativo    388
Name: count, dtype: int64


,review,sentimiento,__index_level_0__
0,Muy buena vista desde la habitaciion.Buen hotel.,positivo,7505
1,"Es un hotel increible. Sus jardines, sus patio...",positivo,3963
2,Me gustó el servicio. igual se escuchaba todo ...,neutro,6088


In [5]:
# Prueba rápida (requiere Ollama + llama3)
ejemplo = str(df["review"].iloc[0])
real = df["sentimiento"].iloc[0]
pred, uso = predecir_sentimiento(ejemplo, return_usage=True)
print("Real:", real, "| Predicho:", pred)
print("Tokens:", uso)

Real: positivo | Predicho: positivo
Tokens: {'prompt_eval_count': 216, 'eval_count': 9, 'total_tokens': 225}


### Procesar lote

- `N_MAX=None` procesa todo el conjunto de validación.
- `OFFSET` salta las primeras filas (útil para reanudar).
- Resultados: `sentimiento_predicho`, uso de tokens por fila. Opcional: guardar CSV al final.

In [6]:
from tqdm.auto import tqdm

N_MAX = None  # p. ej. 50 para prueba; None = todas
OFFSET = 0

sub = df.iloc[OFFSET:]
if N_MAX is not None:
    sub = sub.head(N_MAX)

rows = []
for pos, (idx, row) in enumerate(tqdm(sub.iterrows(), total=len(sub))):
    texto = str(row["review"]) if pd.notna(row["review"]) else ""
    real = str(row["sentimiento"]).strip().lower()
    pred, uso = predecir_sentimiento(texto, return_usage=True)
    rows.append(
        {
            "idx_csv": idx,
            "__index_level_0__": row.get("__index_level_0__", idx),
            "sentimiento_real": real,
            "sentimiento_predicho": pred,
            "prompt_eval_count": uso["prompt_eval_count"],
            "eval_count": uso["eval_count"],
            "total_tokens": uso["total_tokens"],
        }
    )

out_df = pd.DataFrame(rows)
out_df.to_csv(OUT_PATH, index=False)
print("Guardado:", OUT_PATH)
out_df.head(10)

  0%|          | 0/1462 [00:00<?, ?it/s]

Guardado: C:\Estudio\Maestria\Tesis\sources\data\validation_sentimiento_llama3.csv


,idx_csv,__index_level_0__,sentimiento_real,sentimiento_predicho,prompt_eval_count,eval_count,total_tokens
0,0,7505,positivo,positivo,216,9,225
1,1,3963,positivo,positivo,418,9,427
2,2,6088,neutro,negativo,220,9,229
3,3,3083,neutro,neutro,371,10,381
4,4,787,negativo,negativo,417,9,426
5,5,8386,neutro,neutro,222,10,232
6,6,9095,negativo,negativo,234,9,243
7,7,5172,positivo,positivo,429,9,438
8,8,8966,neutro,neutro,223,10,233
9,9,8959,neutro,neutro,228,10,238


### Métricas: predicho vs real

Filas sin predicción válida (`sentimiento_predicho` nulo) se excluyen del informe o se cuentan como error según prefieras; aquí se excluyen para accuracy / F1.

In [7]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

eval_df = out_df.dropna(subset=["sentimiento_predicho"]).copy()
eval_df = eval_df[eval_df["sentimiento_predicho"].isin(ETIQUETAS_VALIDAS)]
eval_df = eval_df[eval_df["sentimiento_real"].isin(ETIQUETAS_VALIDAS)]

y_true = eval_df["sentimiento_real"]
y_pred = eval_df["sentimiento_predicho"]

labels = ["negativo", "neutro", "positivo"]
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}")
print(f"Muestras evaluadas: {len(eval_df)} / {len(out_df)} (total filas en lote)")
print()
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
print("Matriz de confusión (filas=real, columnas=predicho):")
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"real_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
print(cm_df)

Accuracy: 0.8263
Muestras evaluadas: 1462 / 1462 (total filas en lote)

              precision    recall  f1-score   support

    negativo       0.78      0.94      0.85       388
      neutro       0.92      0.69      0.79       674
    positivo       0.78      0.95      0.86       400

    accuracy                           0.83      1462
   macro avg       0.82      0.86      0.83      1462
weighted avg       0.84      0.83      0.82      1462

Matriz de confusión (filas=real, columnas=predicho):
               pred_negativo  pred_neutro  pred_positivo
real_negativo            364           24              0
real_neutro              103          464            107
real_positivo              2           18            380


In [8]:
# Opcional: filas con predicción inválida o faltante
mal = out_df[out_df["sentimiento_predicho"].isna() | ~out_df["sentimiento_predicho"].isin(ETIQUETAS_VALIDAS)]
print(f"Filas sin etiqueta predicha válida: {len(mal)}")
if len(mal):
    print(mal.head(20).to_string())

Filas sin etiqueta predicha válida: 0
